In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import csv
import numpy as np



## **80% y 20%**

In [ ]:
import random

path = "/content/Datacorpus_SIN_normalizar.csv"

ruta_archivo_entrada = path
ruta_archivo_salida_entrenamiento = 'Entrenamiento.csv'
ruta_archivo_salida_prueba = 'Prueba.csv'

# Lista para almacenar los títulos + contenidos y las secciones
titulos_contenidos = []
secciones = []

with open(ruta_archivo_entrada, 'r', newline='', encoding='utf-8') as archivo_csv:
    lector_csv = csv.reader(archivo_csv)
    next(lector_csv)

    for fila in lector_csv:
        titulo_contenido = f"{fila[1]}, {fila[2]}"
        seccion = fila[3]

        titulos_contenidos.append(titulo_contenido)
        secciones.append(seccion)

# Combinar los datos en una lista de tuplas
datos_combinados = list(zip(titulos_contenidos, secciones))

# Mezclar los datos de manera aleatoria
random.shuffle(datos_combinados)

# Calcular el índice de división
indice_division = int(len(datos_combinados) * 0.8)

# Dividir los datos en entrenamiento y prueba
datos_entrenamiento = datos_combinados[:indice_division]
datos_prueba = datos_combinados[indice_division:]

with open(ruta_archivo_salida_entrenamiento, 'w', newline='', encoding='utf-8') as archivo_csv:
    escritor_csv = csv.writer(archivo_csv)
    # Escribir la fila de encabezados
    escritor_csv.writerow(['Titulo + contenidos', 'section'])

    # Escribir las filas de datos de entrenamiento
    for fila in datos_entrenamiento:
        escritor_csv.writerow(fila)

with open(ruta_archivo_salida_prueba, 'w', newline='', encoding='utf-8') as archivo_csv:
    escritor_csv = csv.writer(archivo_csv)
    escritor_csv.writerow(['Titulo + contenidos', 'section'])

    # Escribir las filas de datos de prueba
    for fila in datos_prueba:
        escritor_csv.writerow(fila)

print(f"Archivos '{ruta_archivo_salida_entrenamiento}' y '{ruta_archivo_salida_prueba}' creados exitosamente.")


Archivos 'Entrenamiento.csv' y 'Prueba.csv' creados exitosamente.


## **Normalizacion**

## ***Normalizacion con spacy***

In [ ]:
pip install --upgrade spacy

In [ ]:
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 11.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import re
import csv
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

# Modelo de spaCy para español
nlp = spacy.load('es_core_news_sm')

def tokenizacion(texto):
    return word_tokenize(texto.lower())

def text_cleaning(texto):
    texto = re.sub(r'\W', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto)
    texto = texto.strip()
    return texto

def eliminar_stop_words(tokens):
    stop_words = set(stopwords.words('spanish'))
    return [word for word in tokens if word not in stop_words]

def lematizacion(texto):
    doc = nlp(texto)
    return [token.lemma_ for token in doc]

def procesar_texto(opcion, texto):
    if opcion == 'A':
        texto_limpio = text_cleaning(texto)
        tokens = tokenizacion(texto_limpio)
        return ' '.join(tokens)  # Text cleaning y tokenización
    elif opcion == 'B':
        tokens = tokenizacion(texto)
        tokens_sin_stopwords = eliminar_stop_words(tokens)
        return ' '.join(tokens_sin_stopwords)  # Tokenización y stop words
    elif opcion == 'C':
        tokens_sin_stopwords = eliminar_stop_words(tokenizacion(texto))
        tokens_lematizados = lematizacion(' '.join(tokens_sin_stopwords))
        return ' '.join(tokens_lematizados)  # Stop words y lematización
    elif opcion == 'D':
        texto_limpio = text_cleaning(texto)
        tokens_lematizados = lematizacion(texto_limpio)
        return ' '.join(tokens_lematizados)  # Text cleaning y lematización
    elif opcion == 'E':
        texto_limpio = text_cleaning(texto)
        tokens = tokenizacion(texto_limpio)
        tokens_sin_stopwords = eliminar_stop_words(tokens)
        return ' '.join(tokens_sin_stopwords)  # Text cleaning, tokenización y stop words
    elif opcion == 'F':
        texto_limpio = text_cleaning(texto)
        tokens = tokenizacion(texto_limpio)
        tokens_lematizados = lematizacion(' '.join(tokens))
        return ' '.join(tokens_lematizados)  # Text cleaning, tokenización y lematización
    elif opcion == 'G':
        texto_limpio = text_cleaning(texto)
        tokens = tokenizacion(texto_limpio)
        tokens_sin_stopwords = eliminar_stop_words(tokens)
        tokens_lematizados = lematizacion(' '.join(tokens_sin_stopwords))
        return ' '.join(tokens_lematizados)  # Text cleaning, tokenización, stop words y lematización

def leer_y_procesar_csv(ruta_archivo, opcion):
    titulos_contenidos = []
    secciones = []

    with open(ruta_archivo, 'r', newline='', encoding='utf-8') as archivo_csv:
        lector_csv = csv.reader(archivo_csv)
        next(lector_csv)  # Saltar la primera fila (títulos)

        for fila in lector_csv:
            titulo_contenido = fila[0]
            seccion = fila[1]
            texto_procesado = procesar_texto(opcion, str(titulo_contenido))

            titulos_contenidos.append(texto_procesado)
            secciones.append(seccion)

    return titulos_contenidos, secciones

def escribir_csv(titulos_contenidos, secciones, nombre_archivo):
    with open(nombre_archivo, 'w', newline='', encoding='utf-8') as archivo_csv:
        escritor_csv = csv.writer(archivo_csv)
        escritor_csv.writerow(['Titulo + contenidos', 'section'])

        for i in range(len(titulos_contenidos)):
            escritor_csv.writerow([titulos_contenidos[i], secciones[i]])

ruta_archivo_entrenamiento = 'Entrenamiento.csv'
ruta_archivo_prueba = 'Prueba.csv'

def procesar_opciones(ruta_archivo):
    opciones = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
    for opcion in opciones:
        nombre_archivo_salida = f'{ruta_archivo.split(".")[0]}_{opcion}.csv'
        titulos_contenidos, secciones = leer_y_procesar_csv(ruta_archivo, opcion)
        escribir_csv(titulos_contenidos, secciones, nombre_archivo_salida)
        print(f"Archivo '{nombre_archivo_salida}' creado exitosamente.")

# Procesar Entrenamiento y Prueba con todas las opciones
procesar_opciones(ruta_archivo_entrenamiento)
procesar_opciones(ruta_archivo_prueba)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Archivo 'Entrenamiento_A.csv' creado exitosamente.
Archivo 'Entrenamiento_B.csv' creado exitosamente.
Archivo 'Entrenamiento_C.csv' creado exitosamente.
Archivo 'Entrenamiento_D.csv' creado exitosamente.
Archivo 'Entrenamiento_E.csv' creado exitosamente.
Archivo 'Entrenamiento_F.csv' creado exitosamente.
Archivo 'Entrenamiento_G.csv' creado exitosamente.
Archivo 'Prueba_A.csv' creado exitosamente.
Archivo 'Prueba_B.csv' creado exitosamente.
Archivo 'Prueba_C.csv' creado exitosamente.
Archivo 'Prueba_D.csv' creado exitosamente.
Archivo 'Prueba_E.csv' creado exitosamente.
Archivo 'Prueba_F.csv' creado exitosamente.
Archivo 'Prueba_G.csv' creado exitosamente.


## **Representacion Binarizada, Frecuencia, TF-IDF, enbeddings**

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec
import numpy as np

# Cargar datos de dos CSVs con manejo de codificación
def load_csvs(file_path1, file_path2):
    df1 = pd.read_csv(file_path1, encoding='utf-8')
    df2 = pd.read_csv(file_path2, encoding='utf-8')
    return df1, df2

# Representación binarizada
def binarize_text(text_data):
    vectorizer = CountVectorizer(binary=True)  # Solo unigramas
    binarized_matrix = vectorizer.fit_transform(text_data)
    return binarized_matrix.toarray(), vectorizer.get_feature_names_out()

# Representación por frecuencia
def frequency_text(text_data):
    vectorizer = CountVectorizer()  # Solo unigramas
    frequency_matrix = vectorizer.fit_transform(text_data)
    return frequency_matrix.toarray(), vectorizer.get_feature_names_out()

# Representación TF-IDF
def tfidf_text(text_data):
    vectorizer = TfidfVectorizer()  # Solo unigramas
    tfidf_matrix = vectorizer.fit_transform(text_data)
    return tfidf_matrix.toarray(), vectorizer.get_feature_names_out()

# Representación usando embeddings
#def embeddings_text(text_data):
   # tokenized_text = [text.split() for text in text_data]
    #model = Word2Vec(sentences=tokenized_text, vector_size=100, window=5, min_count=1, workers=4)
   # embeddings_matrix = [np.mean([model.wv[word] for word in words if word in model.wv] or [np.zeros(100)], axis=0) for words in tokenized_text]
   # return embeddings_matrix, None

# Función principal para aplicar transformaciones a dos archivos
def apply_transformation(file_path1, file_path2, transformation, output_path1, output_path2):
    df1, df2 = load_csvs(file_path1, file_path2)
    text_column = 'Titulo + contenidos'  # Usando el nombre correcto de la columna
    section_column = 'section'  # Usando el nombre correcto de la columna

    text_data1 = df1[text_column].tolist()
    text_data2 = df2[text_column].tolist()
    combined_text_data = text_data1 + text_data2

    if transformation == 'binarized':
        result, feature_names = binarize_text(combined_text_data)
    elif transformation == 'frequency':
        result, feature_names = frequency_text(combined_text_data)
    elif transformation == 'tfidf':
        result, feature_names = tfidf_text(combined_text_data)
    else:
        raise ValueError("Transformación no válida. Elija entre 'binarized', 'frequency', 'tfidf'.")

    # Dividir los resultados de vuelta en los dos conjuntos de datos originales
    result1 = result[:len(df1)]
    result2 = result[len(df1):]

    # Crear DataFrames con los resultados
    if feature_names is not None:
        result_df1 = pd.DataFrame(result1, columns=feature_names)
        result_df2 = pd.DataFrame(result2, columns=feature_names)
    else:
        result_df1 = pd.DataFrame(result1)
        result_df2 = pd.DataFrame(result2)

    # Añadir la columna de sección original
    result_df1[section_column] = df1[section_column].values
    result_df2[section_column] = df2[section_column].values

    # Guardar los resultados en nuevos archivos CSV
    result_df1.to_csv(output_path1, index=False)
    result_df2.to_csv(output_path2, index=False)

# Función para procesar todas las opciones
def procesar_todas_opciones():
    opciones = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
    transformaciones = ['binarized', 'frequency', 'tfidf']

    for opcion in opciones:
        for transformacion in transformaciones:
            file_path1 = f'Entrenamiento_{opcion}.csv'
            file_path2 = f'Prueba_{opcion}.csv'
            output_path1 = f'train_{opcion}_{transformacion}.csv'
            output_path2 = f'test_{opcion}_{transformacion}.csv'
            apply_transformation(file_path1, file_path2, transformacion, output_path1, output_path2)
            print(f"Los datos {output_path1} y {output_path2} han sido transformados exitosamente.")

# Ejemplo de uso
if __name__ == "__main__":
    procesar_todas_opciones()



Los datos train_A_binarized.csv y test_A_binarized.csv han sido transformados exitosamente.
Los datos train_A_frequency.csv y test_A_frequency.csv han sido transformados exitosamente.
Los datos train_A_tfidf.csv y test_A_tfidf.csv han sido transformados exitosamente.
Los datos train_B_binarized.csv y test_B_binarized.csv han sido transformados exitosamente.
Los datos train_B_frequency.csv y test_B_frequency.csv han sido transformados exitosamente.
Los datos train_B_tfidf.csv y test_B_tfidf.csv han sido transformados exitosamente.
Los datos train_C_binarized.csv y test_C_binarized.csv han sido transformados exitosamente.
Los datos train_C_frequency.csv y test_C_frequency.csv han sido transformados exitosamente.
Los datos train_C_tfidf.csv y test_C_tfidf.csv han sido transformados exitosamente.
Los datos train_D_binarized.csv y test_D_binarized.csv han sido transformados exitosamente.
Los datos train_D_frequency.csv y test_D_frequency.csv han sido transformados exitosamente.
Los datos tr

Metodo uno, NAIVE BAYES

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

def load_data(train_file, test_file):
    train_data = pd.read_csv(train_file)
    test_data = pd.read_csv(test_file)
    return train_data, test_data

# Función para entrenar y evaluar el clasificador
def train_and_evaluate(train_data, test_data):
    # Separar características y etiquetas
    X_train = train_data.drop(columns=['section'])
    y_train = train_data['section']
    X_test = test_data.drop(columns=['section'])
    y_test = test_data['section']

    # Definir y entrenar el clasificador
    classifier = MultinomialNB()
    classifier.fit(X_train, y_train)

    # Predecir las clases en el conjunto de prueba
    predicted = classifier.predict(X_test)

    # Generar el reporte de clasificación
    report = classification_report(y_test, predicted, output_dict=True, zero_division=0)

    # Extraer F1-score macro average y weighted average
    macro_f1 = report['macro avg']['f1-score']
    weighted_f1 = report['weighted avg']['f1-score']
    print(f"F1-score (macro avg): {macro_f1}")
    print(f"F1-score (weighted avg): {weighted_f1}")


if __name__ == "__main__":
    train_file = 'train_C_binarized.csv'
    test_file = 'test_C_binarized.csv'
    # Cargar los datos
    train_data, test_data = load_data(train_file, test_file)

    # Entrenar y evaluar el clasificador
    train_and_evaluate(train_data, test_data)

F1-score (macro avg): 0.6425308990526382
F1-score (weighted avg): 0.8247373667663525


Metodo dos, REGRESION LINEAL

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

def load_data(train_file, test_file):
    train_data = pd.read_csv(train_file)
    test_data = pd.read_csv(test_file)
    return train_data, test_data

# Función para entrenar y evaluar el clasificador de regresión logística
def train_and_evaluate(train_data, test_data):
    # Separar características y etiquetas
    X_train = train_data.drop(columns=['section'])
    y_train = train_data['section']
    X_test = test_data.drop(columns=['section'])
    y_test = test_data['section']

    # Entrenamiento de clasificador de regresión logística
    classifier = LogisticRegression(max_iter=1000)  # Ajustar max_iter
    classifier.fit(X_train, y_train)

    # Predecir las clases en el conjunto de prueba
    predicted = classifier.predict(X_test)

    # Generar el reporte de clasificación
    report = classification_report(y_test, predicted, output_dict=True)

    # Extraer F1-score macro promedio y promedio ponderado
    macro_f1 = report['macro avg']['f1-score']
    weighted_f1 = report['weighted avg']['f1-score']
    print(f"F1-score (macro avg): {macro_f1}")
    print(f"F1-score (weighted avg): {weighted_f1}")

if __name__ == "__main__":
    train_file = 'train_A_binarized.csv'
    test_file = 'test_A_binarized.csv'

    # Cargar los datos
    train_data, test_data = load_data(train_file, test_file)

    # Entrenar y evaluar el clasificador de regresión logística
    train_and_evaluate(train_data, test_data)


F1-score (macro avg): 0.42175084964256415
F1-score (weighted avg): 0.6284620492988007


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


## **Metodo tres, SVM**

In [ ]:

from sklearn.svm import SVC
from sklearn.metrics import classification_report

# Función para cargar datos desde archivos CSV
def load_data(train_file, test_file):
    train_data = pd.read_csv(train_file)
    test_data = pd.read_csv(test_file)
    return train_data, test_data

# Función para entrenar y evaluar el clasificador SVM
def train_and_evaluate(train_data, test_data):
    # Separar características y etiquetas
    X_train = train_data.drop(columns=['section'])
    y_train = train_data['section']
    X_test = test_data.drop(columns=['section'])
    y_test = test_data['section']

    #clasificador SVM

    #classifier = SVC()
    #classifier = SVC(kernel='poly') # Kernel Polinomial
    #classifier = SVC(kernel='rbf') # Kernel Radial Basis Function (RBF)
    #classifier = SVC(kernel='sigmoid') # Kernel Sigmoidal
    classifier = SVC(kernel='linear')
    classifier.fit(X_train, y_train)

    # Predecir las clases en el conjunto de prueba
    predicted = classifier.predict(X_test)

    # Generar el reporte de clasificación
    report = classification_report(y_test, predicted, output_dict=True)

    # Extraer las métricas macro y weighted avg
    macro_f1 = report['macro avg']['f1-score']
    weighted_f1 = report['weighted avg']['f1-score']
    print(f"F1-score (macro avg): {macro_f1}")
    print(f"F1-score (weighted avg): {weighted_f1}\n\n")

# Ejemplo de uso
if __name__ == "__main__":
    train_file = 'train_C_tfidf.csv'
    test_file = 'test_C_tfidf.csv'

    # Cargar los datos
    train_data, test_data = load_data(train_file, test_file)

    # Entrenar y evaluar el clasificador SVM
    train_and_evaluate(train_data, test_data)


F1-score (macro avg): 0.13529411764705881
F1-score (weighted avg): 0.34575163398692804




/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
